<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/02_weather_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [ ]:
!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git

Cloning into 'AstroPlanner-AI'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 39 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 21.79 KiB | 656.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.


In [ ]:
!pip install requests -q

import sys, os, json, requests
from datetime import date as date_type, datetime, timedelta
from typing import Optional
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'

Mounted at /content/drive


In [ ]:
%cd AstroPlanner-AI/src

/content/AstroPlanner-AI/src


In [ ]:
from models import UserProfile

In [ ]:
import requests
from datetime import date as date_type, datetime
from typing import Optional

In [ ]:
from models import WeeklyPlanRequest, UserProfile

with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

print(plan_request.model_dump_json(indent=2))

{
  "user": {
    "name": "Arfa",
    "latitude": 33.223,
    "longitude": 32.44,
    "experience_level": "beginner",
    "telescope": {
      "aperture_mm": 64.0,
      "focal_length_mm": 323.0,
      "focal_ratio": null
    },
    "camera": {
      "sensor_width_mm": 32.0,
      "sensor_height_mm": 23.0,
      "pixel_size_um": 3.0
    },
    "mount": {
      "type": "alt-az",
      "goto_capable": true
    }
  },
  "notes": null
}


### Open-Meteo fetch (general weather)

In [ ]:
def fetch_open_meteo_forecast(latitude: float, longitude: float, target_date: str) -> dict:
    """
    Fetches daily and hourly weather forecast from Open-Meteo for the given
    coordinates and date. No API key required. Returns cloud cover, humidity,
    wind speed, temperature, and precipitation probability — the general
    weather baseline. Seeing/transparency (astronomy-specific) are NOT
    available here and come from fetch_7timer_astro instead.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "cloudcover,relative_humidity_2m,wind_speed_10m,temperature_2m,precipitation_probability",
        "start_date": target_date,
        "end_date": target_date,
        "timezone": "auto",
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    hourly = data.get("hourly", {})
    times = hourly.get("time", [])

    target_hour = f"{target_date}T22:00"
    if target_hour in times:
        idx = times.index(target_hour)
    elif times:
        idx = len(times) // 2
    else:
        idx = None

    return {
        "source": "open-meteo",
        "date": target_date,
        "reference_time": times[idx] if idx is not None else None,
        "cloud_cover_pct": hourly.get("cloudcover", [None])[idx] if idx is not None else None,
        "humidity_pct": hourly.get("relative_humidity_2m", [None])[idx] if idx is not None else None,
        "wind_speed_kmh": hourly.get("wind_speed_10m", [None])[idx] if idx is not None else None,
        "temperature_c": hourly.get("temperature_2m", [None])[idx] if idx is not None else None,
        "precipitation_probability_pct": hourly.get("precipitation_probability", [None])[idx] if idx is not None else None,
    }

### 7Timer astro fetch (seeing/transparency)

In [ ]:
def fetch_7timer_astro(latitude: float, longitude: float, target_date: str) -> Optional[dict]:
    """
    Fetches the astronomy-specific forecast from 7Timer (seeing, transparency,
    cloud cover) for the given coordinates. Only forecasts ~3 days ahead from
    today; returns None for dates beyond that window or on any API failure,
    so callers degrade gracefully to weather-only data.
    """
    today = date_type.today()
    requested = date_type.fromisoformat(target_date)
    days_out = (requested - today).days

    if days_out < 0 or days_out > 3:
        return None

    url = "http://www.7timer.info/bin/api.pl"
    params = {"lon": longitude, "lat": latitude, "product": "astro", "output": "json"}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except (requests.RequestException, ValueError):
        return None

    series = data.get("dataseries", [])
    init_str = data.get("init")
    if not series or not init_str:
        return None

    init_dt = datetime.strptime(init_str, "%Y%m%d%H")
    target_dt = datetime.combine(requested, datetime.min.time()).replace(hour=22)

    best_entry = min(
        series,
        key=lambda e: abs((init_dt + timedelta(hours=e["timepoint"])) - target_dt),
    )

    return {
        "source": "7timer",
        "date": target_date,
        "cloud_cover_index": best_entry.get("cloudcover"),
        "seeing_index": best_entry.get("seeing"),
        "transparency_index": best_entry.get("transparency"),
    }

### Combine into a sky quality score

In [ ]:
def compute_sky_quality_score(open_meteo_data: dict, astro_data: Optional[dict]) -> dict:
    """
    Combines general weather with astronomy-specific seeing/transparency
    (when available) into a 0-100 sky quality score and plain-language
    verdict. Pure rule-based scoring — no LLM involved.
    """
    score = 100
    reasons = []

    cloud_cover = open_meteo_data.get("cloud_cover_pct")
    if cloud_cover is not None:
        score -= cloud_cover * 0.6
        if cloud_cover > 70:
            reasons.append(f"Heavy cloud cover ({cloud_cover}%)")
        elif cloud_cover > 30:
            reasons.append(f"Moderate cloud cover ({cloud_cover}%)")

    precip = open_meteo_data.get("precipitation_probability_pct")
    if precip is not None and precip > 30:
        score -= precip * 0.3
        reasons.append(f"Precipitation risk ({precip}%)")

    wind = open_meteo_data.get("wind_speed_kmh")
    if wind is not None and wind > 25:
        score -= min((wind - 25), 30)
        reasons.append(f"High wind ({wind} km/h) — may cause telescope vibration")

    humidity = open_meteo_data.get("humidity_pct")
    if humidity is not None and humidity >= 85:  # fixed: was > 85, now >= 85
        score -= 10
        reasons.append(f"High humidity ({humidity}%) — dew risk on optics")

    if astro_data:
        seeing = astro_data.get("seeing_index")
        transparency = astro_data.get("transparency_index")
        if seeing is not None:
            score -= (seeing - 1) * 5
            if seeing >= 6:
                reasons.append("Poor atmospheric seeing — fine detail will be blurry")
        if transparency is not None:
            score -= (transparency - 1) * 5
            if transparency >= 6:
                reasons.append("Poor sky transparency — faint objects harder to see")

    score = max(0, min(100, round(score)))

    if score >= 80:
        verdict = "Excellent"
    elif score >= 60:
        verdict = "Good"
    elif score >= 40:
        verdict = "Fair"
    elif score >= 20:
        verdict = "Poor"
    else:
        verdict = "Not recommended"

    return {
        "sky_quality_score": score,
        "verdict": verdict,
        "reasons": reasons if reasons else ["Clear conditions, no major concerns"],
    }

### Top-level function: this is what becomes the LangChain tool

In [ ]:
def get_weekly_sky_conditions(user_profile: "UserProfile", start_date: date_type) -> list[dict]:
    """
    Generates a 7-day sky conditions outlook (start_date through
    start_date+6) for the user's location. Days 0-3 include full data:
    weather + 7Timer seeing/transparency. Days 4-6 are 'extended_outlook':
    weather only, since no free seeing/transparency forecast extends
    that far.

    Anchored to start_date (plan_request.generated_at, fixed once in
    notebook 01) rather than calling today() independently — otherwise
    running this notebook on a different calendar day than notebook 03
    produces two 7-day windows that don't fully overlap, silently
    dropping nights where they disagree.
    """
    lat = user_profile.latitude
    lon = user_profile.longitude

    weekly_plan = []
    for offset in range(7):
        day = start_date + timedelta(days=offset)
        day_str = day.isoformat()

        weather = fetch_open_meteo_forecast(lat, lon, day_str)
        astro = fetch_7timer_astro(lat, lon, day_str)
        quality = compute_sky_quality_score(weather, astro)

        weekly_plan.append({
            "date": day_str,
            "day_offset": offset,
            "data_confidence": "full" if astro is not None else "extended_outlook",
            "weather": weather,
            "astro_conditions": astro,
            "sky_quality": quality,
        })

    return weekly_plan

###Test it end-to-end

In [ ]:
weekly_conditions = get_weekly_sky_conditions(plan_request.user)

with open(f'{DATA_DIR}/weekly_sky_conditions.json', 'w') as f:
    json.dump(weekly_conditions, f, indent=2, default=str)

print(f"Saved to {DATA_DIR}/weekly_sky_conditions.json")
print(json.dumps(weekly_conditions, indent=2, default=str))

Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_sky_conditions.json
[
  {
    "date": "2026-07-01",
    "day_offset": 0,
    "data_confidence": "full",
    "weather": {
      "source": "open-meteo",
      "date": "2026-07-01",
      "reference_time": "2026-07-01T22:00",
      "cloud_cover_pct": 20,
      "humidity_pct": 78,
      "wind_speed_kmh": 27.9,
      "temperature_c": 26.2,
      "precipitation_probability_pct": 0
    },
    "astro_conditions": {
      "source": "7timer",
      "date": "2026-07-01",
      "cloud_cover_index": 1,
      "seeing_index": 6,
      "transparency_index": 4
    },
    "sky_quality": {
      "sky_quality_score": 45,
      "verdict": "Fair",
      "reasons": [
        "High wind (27.9 km/h) \u2014 may cause telescope vibration",
        "Poor atmospheric seeing \u2014 fine detail will be blurry"
      ]
    }
  },
  {
    "date": "2026-07-02",
    "day_offset": 1,
    "data_confidence": "full",
    "weather": {
      "source": "open-meteo",
    